# Integrating bayesflow-trained likelihoods into HSSM (NLE via ONNX)

This notebook covers the bayesflow version of the ONNX-based NLE integration workflow in HSSM. Once you've trained a [bayesflow](https://github.com/bayesflow-org/bayesflow) `ContinuousApproximator` (NLE) or `RatioApproximator` (NRE), you can export it to a single ONNX file and hand the path to HSSM. From HSSM's side, the file is consumed identically to a LAN export or an sbi export — same `loglik="file.onnx"` gesture, no library-specific glue.

Two paths into HSSM exist, side by side:

| Path | Source | Mechanism | When to use |
|---|---|---|---|
| `loglik="file.onnx"` | sbi or bayesflow | ONNX file, framework-agnostic | Portability, sharing trained surrogates |
| `loglik=<jax_callable>` | bayesflow (this tutorial's sibling) | In-memory JAX callable | Fast iteration during model development |

See [`bayesflow_lre_integration.ipynb`](./bayesflow_lre_integration.ipynb) for the JAX-callable path. This notebook covers the ONNX path.

## Part 1 — Setup

**Critical**: `KERAS_BACKEND=torch` must be set *before* importing `keras` or `bayesflow`. `torch.onnx.export` cannot trace a JAX-backed Keras model. On Apple silicon also pin `KERAS_TORCH_DEVICE=cpu` to avoid the orthogonal initializer's missing MPS op.

**Note on the two ONNX-related setup lines below** (`jax_enable_x64` and the `jaxonnxruntime` strict-mode relax): these are the same workarounds that HSSM PR #964 plans to auto-handle inside `hssm.distribution_utils.onnx2jax`. Until that PR lands on `main`, this tutorial sets them explicitly so it runs standalone. Once #964 merges, both lines can be deleted — HSSM will take care of them on import.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"
os.environ["KERAS_TORCH_DEVICE"] = "cpu"

import jax

# x64 BEFORE other JAX-touching imports; ONNX graphs from torch.onnx.export
# carry int64 shape/index tensors that get silently truncated under JAX's
# default int32 mode, producing wrong log-prob values inside HSSM.
jax.config.update("jax_enable_x64", True)

# Relax jaxonnxruntime's default strict mode on Reshape shape arguments —
# torch.onnx.export emits them as Constant nodes, not initializers, which the
# strict default rejects. Safe because the shapes are genuinely constant.
# HSSM PR #964 sets this automatically inside hssm.distribution_utils.onnx2jax;
# until that PR lands on main, we set it here.
from jaxonnxruntime import config as _jaxonnx_config
_jaxonnx_config.update("jaxort_only_allow_initializers_as_static_args", False)

import tempfile  # noqa: F401 — available for the Part 4 override example
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bayesflow as bf
import keras
from bayesflow.datasets import OfflineDataset
from bayesflow.networks.inference.coupling.transforms import AffineTransform
from ssms.basic_simulators.simulator import simulator

import hssm
from lanfactory.onnx import transform_bayesflow_to_onnx

print("keras backend:", keras.backend.backend())
print("bayesflow:    ", bf.__version__)
print("hssm:         ", hssm.__version__)

## Part 2 — Simulate observed DDM data

We use [`ssm-simulators`](https://github.com/AlexanderFengler/ssm-simulators) for ground-truth DDM samples at a known parameter vector. HSSM consumes a `DataFrame` with `rt` and `response` columns.

In [ ]:
DDM_PARAM_NAMES = ["v", "a", "z", "t"]
DDM_PARAM_LOW = np.array([-2.0, 0.6, 0.3, 0.1], dtype=np.float32)
DDM_PARAM_HIGH = np.array([2.0, 1.8, 0.7, 0.5], dtype=np.float32)
TRUE_THETA = np.array([0.5, 1.2, 0.5, 0.25], dtype=np.float32)
N_OBS = 500

out = simulator(theta=TRUE_THETA[None, :], model="ddm", n_samples=N_OBS)
obs_data = pd.DataFrame({
    "rt": out["rts"].squeeze().astype(np.float32),
    "response": out["choices"].squeeze().astype(np.float32),
})
obs_data.head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for choice, label in [(1.0, "choice +1"), (-1.0, "choice -1")]:
    rts = obs_data.loc[obs_data["response"] == choice, "rt"]
    ax.hist(rts, bins=40, alpha=0.5, label=label)
ax.set_xlabel("reaction time")
ax.set_ylabel("count")
ax.legend()
ax.set_title(f"Observed DDM data at θ={dict(zip(DDM_PARAM_NAMES, TRUE_THETA))}")
plt.tight_layout()

## Part 3 — Train a bayesflow CouplingFlow NLE on DDM simulations

The CouplingFlow setup below is the v1 ONNX-friendly configuration documented in [LANfactory's `exporting_bayesflow_models.md`](https://github.com/lnccbrown/LANFactory/blob/main/docs/exporting_bayesflow_models.md). Three opinionated choices that aren't bayesflow's defaults:

- `permutation=None` — `FixedPermutation` uses `keras.ops.take`, which exports as `aten::ravel`, unsupported in ONNX opset 17/20.
- `transform=AffineTransform(clamp=False)` (explicit instance) — default `clamp=True` emits `aten::asinh`. Also, `bf.networks.CouplingFlow(..., transform_kwargs={"clamp": False})` silently drops the kwarg (upstream bug); pass an instance.
- `activation="silu"` — default `"hard_silu"` exports as a single fused `HardSwish` op that jaxonnxruntime can't run. Real SiLU decomposes to `Sigmoid` + `Mul`.

In [ ]:
keras.utils.set_random_seed(0)
rng = np.random.default_rng(0)

N_TRAIN = 20_000
theta_train = rng.uniform(
    DDM_PARAM_LOW, DDM_PARAM_HIGH, size=(N_TRAIN, len(DDM_PARAM_NAMES))
).astype(np.float32)

# One trial per (θ_i) — NLE convention: each row is (θ_i, x_i).
x_train = np.empty((N_TRAIN, 2), dtype=np.float32)
for i, th in enumerate(theta_train):
    out = simulator(theta=th[None, :], model="ddm", n_samples=1)
    x_train[i, 0] = out["rts"].squeeze()
    x_train[i, 1] = out["choices"].squeeze()

print("theta_train:", theta_train.shape, "  x_train:", x_train.shape)

In [ ]:
approximator = bf.ContinuousApproximator(
    inference_network=bf.networks.CouplingFlow(
        depth=6,
        subnet_kwargs={"widths": (64, 64), "activation": "silu", "dropout": None},
        permutation=None,
        use_actnorm=False,
        transform=AffineTransform(clamp=False),
    ),
    standardize="inference_variables",  # standardize x (rt, choice)
)
approximator.build({
    "inference_variables": (None, 2),
    "inference_conditions": (None, len(DDM_PARAM_NAMES)),
})
approximator.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4))

dataset = OfflineDataset(
    data={
        "inference_variables": x_train,
        "inference_conditions": theta_train,
    },
    batch_size=256,
    adapter=None,  # MUST be identity for ONNX export; we use only the in-network Standardize layer
)
history = approximator.fit(dataset=dataset, epochs=50, verbose=1)

## Part 4 — Export the trained approximator to ONNX

One call. The exporter raises clearly if any v1 constraint is violated (wrong `KERAS_BACKEND`, non-identity adapter, missing `inference_network`).

**Where to write the file.** The cell below sets `ARTIFACT_DIR` to `~/bayesflow_onnx_tutorial/` — outside the HSSM repo, so re-running the notebook doesn't leave artifacts in your working tree. Change it to any path you want to keep the trained ONNX around for downstream work; set it to `tempfile.mkdtemp()` for an ephemeral demo run.

In [ ]:
# User-configurable: where the .onnx file lands. Default is outside the HSSM
# repo so notebook re-runs don't pollute the working tree.
# Override examples:
#   ARTIFACT_DIR = Path("/path/to/my/project/onnx")          # keep nearby
#   ARTIFACT_DIR = Path(tempfile.mkdtemp())                  # ephemeral
ARTIFACT_DIR = Path.home() / "bayesflow_onnx_tutorial"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
onnx_path = ARTIFACT_DIR / "ddm_nle.onnx"

transform_bayesflow_to_onnx(
    approximator,
    str(onnx_path),
    mode="nle",
    example_theta_dim=len(DDM_PARAM_NAMES),
    example_x_dim=2,
)
print(f"wrote {onnx_path}  ({onnx_path.stat().st_size:,} bytes)")

## Part 5 — Hand the ONNX to HSSM

The user gesture is identical to the sbi path and the LAN-MLP path. HSSM detects the `.onnx` extension, loads it via `jaxonnxruntime`, vmaps over trials, and wires it into a PyMC `Distribution`.

In [ ]:
model_nle = hssm.HSSM(
    data=obs_data,
    model="ddm",
    loglik_kind="approx_differentiable",
    loglik=str(onnx_path),
    p_outlier=0,
)
model_nle

In [ ]:
idata_nle = model_nle.sample(
    sampler="numpyro",
    draws=1000,
    tune=1000,
    chains=2,
    target_accept=0.9,
    mp_ctx="spawn",
)

In [ ]:
summary_nle = az.summary(idata_nle, var_names=DDM_PARAM_NAMES)
summary_nle

In [ ]:
az.plot_trace(idata_nle, var_names=DDM_PARAM_NAMES)
plt.tight_layout()

## Part 6 — Ground-truth posterior via HSSM's analytical DDM

DDM has a closed-form likelihood (Navarro & Fuss). Comparing the bayesflow-NLE posterior to this gives a fairness check: any drift comes from the neural approximation, not from the inference machinery.

In [ ]:
model_analytical = hssm.HSSM(
    data=obs_data,
    model="ddm",
    p_outlier=0,
    # default loglik_kind here is the analytical Navarro & Fuss path
)
idata_analytical = model_analytical.sample(
    sampler="numpyro",
    draws=1000,
    tune=1000,
    chains=2,
    target_accept=0.9,
)
summary_analytical = az.summary(idata_analytical, var_names=DDM_PARAM_NAMES)
summary_analytical

## Part 7 — Posterior comparison: bayesflow NLE vs analytical

Overlay marginals and mark the true values. If the bayesflow NLE was trained well, the two posterior densities should agree closely.

In [ ]:
fig, axes = plt.subplots(1, len(DDM_PARAM_NAMES), figsize=(16, 4))
for ax, name, true_val in zip(axes, DDM_PARAM_NAMES, TRUE_THETA):
    az.plot_kde(
        idata_analytical.posterior[name].values.flatten(),
        plot_kwargs={"color": "black", "linestyle": "--", "label": "analytical"},
        ax=ax,
    )
    az.plot_kde(
        idata_nle.posterior[name].values.flatten(),
        plot_kwargs={"color": "tab:blue", "label": "bayesflow NLE"},
        ax=ax,
    )
    ax.axvline(true_val, color="red", lw=1, label="truth")
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.tight_layout()

## Summary

- **User gesture is the same as sbi or LAN-MLP**: `hssm.HSSM(loglik="file.onnx", loglik_kind="approx_differentiable")`. HSSM doesn't need to know which framework trained the surrogate.
- **The v1 constraints on the bayesflow side** (CouplingFlow with `permutation=None`, explicit `AffineTransform(clamp=False)`, `silu` not `hard_silu` activation, identity adapter, `KERAS_BACKEND=torch` at export time) are all enforced or documented by `lanfactory.onnx.transform_bayesflow_to_onnx`. The exporter raises clearly when a constraint is violated.
- **For an in-memory JAX-callable alternative** (no ONNX file, faster iteration during model development), see [`bayesflow_lre_integration.ipynb`](./bayesflow_lre_integration.ipynb).

Out of v1 scope (tracked as future work):

- MNLE-style discrete + continuous mixed observations
- Non-identity bayesflow Adapters (would require either baking the tensor-able subset into the ONNX graph, or shipping the adapter spec alongside the ONNX file)
- Transformer / attention summary networks (LayerNorm + dynamic-shape ops)
- FlowMatching / DiffusionModel / ConsistencyModel inference networks (`log_prob` requires ODE integration)